# Module 1 — Emotion vector extraction (Gemma-2-9B-IT)

**v2 §3 Methodology.** Extract 4 emotion vectors (desperation, calm, sad, angry) from Gemma-2-9B-IT residual stream using the Anthropic / Sofroniew-Lindsey 2026 protocol.

**Pipeline:**
1. Generate stories with Claude Opus 4.7 (20 per emotion, stratified across narrative contexts).
2. Run each story through Gemma-2-9B-IT; capture residual-stream activations at layer 28 (~2/3 depth).
3. Average from token 50 onward → per-story vector.
4. Per-emotion mean across stories.
5. Subtract cross-emotion mean.
6. Project out top PCs of neutral corpus (50% variance).
7. ℓ₂-normalize.

**Run on:** Colab Pro+ A100 or Vast/Lambda A100 80GB. Gemma-2-9B in bf16 = ~18GB VRAM; A100 fits comfortably.

**Cost estimate:** stories generation ~$4 (Anthropic API). Activation capture ~2–4 GPU-hr on A100. Total ~$15.

**Prerequisite:** `sanity_test.ipynb` must pass first.

## Cell 1 — env setup

In [ ]:
# Colab block — uncomment
# from google.colab import userdata
# import os
# os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
# os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
# !git clone https://github.com/<user>/desperation-circuit.git
# %cd desperation-circuit

!bash scripts/setup_env.sh

## Cell 2 — generate stories (cheap, run first)

Generates 4 emotions × 20 stories = 80 stories, plus a 20-story neutral corpus for PC project-out. ~5 min on Anthropic API. Resumable: existing files are skipped.

In [ ]:
import sys
sys.path.insert(0, '.')

from pathlib import Path
from src.generate_stories import generate_emotion_corpus, generate_all
from src.lib.config import load_config

cfg = load_config()
data_dir = Path(cfg['paths']['data_dir'])

# main emotion corpus (4 emotions x 20 stories)
generate_all()

# neutral corpus for PC project-out (separate; not in main config emotions list)
generate_emotion_corpus(
    emotion='neutral',
    n_stories=20,
    out_dir=data_dir / 'stories' / 'neutral',
    target_words=cfg['extraction']['story_target_word_count'],
)

## Cell 3 — spot-check generated stories

Read a couple to confirm the emotion is unmistakable. If Opus's stories are too subtle or too literal, tighten the prompt in `src/generate_stories.py` before extracting vectors.

In [ ]:
for emotion in cfg['extraction']['emotions']:
    sample = (data_dir / 'stories' / emotion / '000.txt').read_text(encoding='utf-8')
    print(f'\n=== {emotion.upper()} (story 000, first 400 chars) ===')
    print(sample[:400])

## Cell 4 — load Gemma-2-9B-IT

In [ ]:
from src.lib.model_load import load_gemma

model, tokenizer = load_gemma(variant='primary')
print(f'loaded {model.config._name_or_path}, n_layers={model.config.num_hidden_layers}, d_model={model.config.hidden_size}')

## Cell 5 — run extraction

In [ ]:
from src.extract_vectors import extract_all

results = extract_all(model, tokenizer)
for emotion, r in results.items():
    print(f'{emotion}: vector shape={r.vector.shape}, n_stories_used={r.n_stories_used}')

## Cell 6 — sanity check the vectors

Cosine matrix between emotion vectors. v2 §3 names sad and angry as negative-valence controls and calm as the directional/inverse control. If desperation's cosine with any control exceeds ~0.5, the controls aren't clean and we need to re-examine story generation before moving to Module 2.

In [ ]:
import json
import numpy as np
import pandas as pd

log = json.loads((Path(cfg['paths']['outputs_dir']) / 'm1_vectors' / 'extraction_log.json').read_text())
print('extraction log:')
print(json.dumps(log, indent=2))

emotions = cfg['extraction']['emotions']
vecs = {e: results[e].vector for e in emotions}
cos = pd.DataFrame(
    {e1: {e2: float(vecs[e1] @ vecs[e2]) for e2 in emotions} for e1 in emotions}
)
print('\npairwise cosines:')
cos

## Outputs

- `data/stories/{emotion}/*.txt` — 4 × 20 emotion stories + 20 neutral
- `outputs/m1_vectors/{emotion}.npy` — ℓ₂-normalized vectors at extraction layer
- `outputs/m1_vectors/extraction_log.json` — protocol params + pairwise cosines + n_stories_used

## Push artifacts to HF Hub

Stories and vectors should land in the private dataset repo so teammates can reproduce. Don't commit `.npy` or `data/stories/*.txt` to git (already gitignored).

```python
from huggingface_hub import HfApi
api = HfApi()
api.upload_folder(
    folder_path='outputs/m1_vectors',
    repo_id=cfg['paths']['hf_artifact_repo'],
    repo_type='dataset',
    path_in_repo='m1_vectors',
)
```

## Next

Once the cosine sanity check looks clean, write `m2_steer_mmlu.ipynb` (steering sweep + MMLU capability check).